### Data extractor 

In [ ]:
import json
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
from plotly.colors import hex_to_rgb
import plotly.graph_objects as go
import plotly.express as px
import tqdm
import ipywidgets as widgets
from IPython.display import display


In [ ]:
def parse_zip_experiments(zip_filepath):
    data = []
    
    with zipfile.ZipFile(zip_filepath, 'r') as archive:
        for filename in tqdm.tqdm(archive.namelist()):
            if filename.endswith('.json'):
                with archive.open(filename) as f:
                    try:
                        exp = json.load(f)
                        
                        params = exp.get("parameters", {})
                        num_agents = params.get("numAgents")
                        num_objects = params.get("numObjects")
                        ratio_random = params.get("ratioRandom", 0) 
                        seed_val = params.get("seed", "Inconnu")
                        
                        policy_val = params.get("selectedPolicy", "Standard")

                        results = exp.get("results", {})
                        
                        solver = results.get("solver", {})
                        solver_time = solver.get("timeUs", 0)
                        solver_score = solver.get("score", 0)
                        solver_metrics = solver.get("metrics", {})
                        
                        mcts = results.get("mcts", {})
                        mcts_tries = mcts.get("tries", [])
                        
                        mcts_total_times = []
                        mcts_final_scores = []
                        
                        mcts_metrics_counts = {
                            "ParetoOptimal": 0, "Prop": 0, 
                            "EF": 0, "EFX": 0, "EF1": 0
                        }
                        
                        for t in mcts_tries:
                            steps = t.get("steps", [])
                            if steps:
                                mcts_total_times.append(sum(step.get("stepTimeUs", 0) for step in steps))
                                mcts_final_scores.append(steps[-1].get("score", 0))
                                final_metrics = steps[-1].get("metrics", {})
                                for metric in mcts_metrics_counts.keys():
                                    if final_metrics.get(metric, False):
                                        mcts_metrics_counts[metric] += 1
                                        
                        # --- NOUVEAU : Calcul des moyennes, mins et maxs ---
                        if mcts_total_times:
                            num_tries = len(mcts_total_times)
                            row_data = {
                                "Agents": num_agents,
                                "Objets": num_objects,
                                "Ratio": ratio_random,
                                "Seed" : seed_val,
                                "Policy": policy_val,
                                "Solveur_Temps_us": solver_time,
                                "Solveur_Score": solver_score,
                                "MCTS_Temps_Moyen_us": sum(mcts_total_times) / num_tries,
                                "MCTS_Temps_Min_us": min(mcts_total_times),
                                "MCTS_Temps_Max_us": max(mcts_total_times),
                                "MCTS_Score_Moyen": sum(mcts_final_scores) / num_tries,
                                "MCTS_Score_Min": min(mcts_final_scores),
                                "MCTS_Score_Max": max(mcts_final_scores)
                            }
                            for metric in mcts_metrics_counts.keys():
                                row_data[f"Solveur_{metric}"] = int(solver_metrics.get(metric, False))
                                row_data[f"MCTS_{metric}"] = mcts_metrics_counts[metric] / num_tries
                            data.append(row_data)
                    except Exception as e:
                        print(f"Erreur lors de la lecture de {filename} : {e}")

    return pd.DataFrame(data)

In [ ]:
def plot_comparisons(df):
    if df.empty:
        print("Aucune donnée trouvée dans le ZIP.")
        return

    # On identifie combien de configurations d'agents différentes tu as
    configurations_agents = df['Agents'].unique()
    configurations_agents.sort()

    # Pour chaque nombre d'agents, on dessine un graphique séparé
    for agents in configurations_agents:
        # On filtre pour ne garder que les données de ce nombre d'agents
        df_filtre = df[df['Agents'] == agents]
        
        # On fait la moyenne par nombre d'objets
        df_grouped = df_filtre.groupby('Objets').mean().reset_index()

        # Si on n'a qu'un seul point (un seul nombre d'objets testé pour ce nombre d'agents), on le signale
        if len(df_grouped) < 2:
            print(f"⚠️ Pas assez de variations d'objets pour tracer une courbe pour {agents} agents.")
            continue

        # Création de la figure
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        fig.suptitle(f"Analyse pour {agents} Agents", fontsize=16, fontweight='bold')

        # --- Courbe 1 : Temps d'exécution (Axe X = Objets) ---
        ax1.plot(df_grouped['Objets'], df_grouped['Solveur_Temps_us'], marker='o', label='Solveur', color='red', linewidth=2)
        ax1.plot(df_grouped['Objets'], df_grouped['MCTS_Temps_Moyen_us'], marker='s', label='MCTS (Moyenne)', color='blue', linewidth=2)
        ax1.set_title("Évolution du Temps d'exécution")
        ax1.set_xlabel("Nombre d'Objets")
        ax1.set_ylabel("Temps (microsecondes)")
        ax1.set_yscale('log') # Conserve l'échelle logarithmique pour les temps exponentiels
        ax1.legend()
        ax1.grid(True, which="both", ls="--")

        # --- Courbe 2 : Score (Axe X = Objets) ---
        ax2.plot(df_grouped['Objets'], df_grouped['Solveur_Score'], marker='o', label='Solveur', color='red', linewidth=2)
        ax2.plot(df_grouped['Objets'], df_grouped['MCTS_Score_Moyen'], marker='s', label='MCTS (Moyenne)', color='blue', linewidth=2)
        ax2.set_title("Évolution des Scores")
        ax2.set_xlabel("Nombre d'Objets")
        ax2.set_ylabel("Score")
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()
        plt.show()

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_comparisons_interactive(df):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    # 1. Options des widgets
    agents_options = ['Tous'] + sorted(df['Agents'].unique().tolist())
    objets_options = ['Tous'] + sorted(df['Objets'].unique().tolist())
    
    if 'Seed' in df.columns:
        seed_options = ['Tous'] + sorted(df['Seed'].unique().tolist())
    else:
        seed_options = ["Non défini"]
        
    # NOUVEAU : Options pour la Politique
    if 'Policy' in df.columns:
        policy_options = ['Tous'] + sorted(df['Policy'].unique().tolist())
    else:
        policy_options = ["Non défini"]

    # 2. Création des widgets
    dropdown_agents = widgets.Dropdown(options=agents_options, value=agents_options[1] if len(agents_options)>1 else 'Tous', description='👥 Agents:', layout={'width': 'max-content'})
    dropdown_objets = widgets.Dropdown(options=objets_options, value='Tous', description='📦 Objets:', layout={'width': 'max-content'})
    dropdown_seed = widgets.Dropdown(options=seed_options, value='Tous', description='🌱 Graine:', layout={'width': 'max-content'})
    # NOUVEAU : Widget Politique
    dropdown_policy = widgets.Dropdown(options=policy_options, value='Tous', description='⚙️ Politique:', layout={'width': 'max-content'})
    
    # Zone d'affichage
    ui = widgets.HBox([dropdown_agents, dropdown_objets, dropdown_seed, dropdown_policy])
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            
            ag = dropdown_agents.value
            ob = dropdown_objets.value
            sd = dropdown_seed.value
            pol = dropdown_policy.value # NOUVEAU
            
            # Filtrage dynamique
            df_filtre = df.copy()
            if ag != 'Tous': df_filtre = df_filtre[df_filtre['Agents'] == ag]
            if ob != 'Tous': df_filtre = df_filtre[df_filtre['Objets'] == ob]
            if sd != 'Tous' and sd != "Non défini": df_filtre = df_filtre[df_filtre['Seed'] == sd]
            if pol != 'Tous' and pol != "Non défini": df_filtre = df_filtre[df_filtre['Policy'] == pol] # NOUVEAU

            if df_filtre.empty:
                print(f"⚠️ Aucune donnée trouvée pour Agents: {ag}, Objets: {ob}, Graine: {sd}, Pol: {pol}.")
                return

            # --- INTELLIGENCE DE L'AXE X ---
            if ag != 'Tous' and ob == 'Tous':
                x_col = 'Objets'
                x_title = "Nombre d'Objets"
            elif ag == 'Tous' and ob != 'Tous':
                x_col = 'Agents'
                x_title = "Nombre d'Agents"
            elif ag == 'Tous' and ob == 'Tous':
                x_col = 'Objets' 
                x_title = "Nombre d'Objets (Moyenne sur tous les agents)"
            else:
                if 'Ratio' in df_filtre.columns and len(df_filtre['Ratio'].unique()) > 1:
                    x_col = 'Ratio'
                    x_title = "Ratio Random"
                elif 'Policy' in df_filtre.columns and pol == 'Tous' and len(df_filtre['Policy'].unique()) > 1:
                    x_col = 'Policy' # Affiche les politiques sur l'axe X !
                    x_title = "Politique d'Exploration"
                elif 'Seed' in df_filtre.columns and sd == 'Tous' and len(df_filtre['Seed'].unique()) > 1:
                    x_col = 'Seed'
                    x_title = "Graine"
                else:
                    x_col = 'Objets'
                    x_title = "Configuration Unique"

            # Calcul des moyennes selon l'axe X choisi (on force numeric_only pour éviter les erreurs avec le texte)
            df_grouped = df_filtre.groupby(x_col).mean(numeric_only=True).reset_index()
            
            # Création de la figure avec deux colonnes
            fig = make_subplots(
                rows=1, cols=2, 
                subplot_titles=("Temps d'exécution", "Scores Absolus")
            )

            # --- Graphique 1 : Temps ---
            fig.add_trace(go.Scatter(
                x=df_grouped[x_col], y=df_grouped['Solveur_Temps_us'],
                mode='lines+markers', name='Solveur (Temps)', line=dict(color='red')
            ), row=1, col=1)
            
            fig.add_trace(go.Scatter(
                x=df_grouped[x_col], y=df_grouped['MCTS_Temps_Moyen_us'],
                mode='lines+markers', name='MCTS Moyenne (Temps)', line=dict(color='blue')
            ), row=1, col=1)

            # --- Graphique 2 : Scores Absolus ---
            fig.add_trace(go.Scatter(
                x=df_grouped[x_col], y=df_grouped['Solveur_Score'],
                mode='lines+markers', name='Solveur (Score)', line=dict(color='red', dash='dot')
            ), row=1, col=2)
            
            fig.add_trace(go.Scatter(
                x=df_grouped[x_col], y=df_grouped['MCTS_Score_Moyen'],
                mode='lines+markers', name='MCTS Moyenne (Score)', line=dict(color='blue', dash='dot')
            ), row=1, col=2)

            # --- Mise en forme ---
            titre = f"Comparaison MCTS vs Solveur — Agents: {ag} | Objets: {ob} | Graine: {sd} | Pol: {pol}"
            fig.update_layout(
                title_text=titre,
                title_font=dict(size=18),
                hovermode="x unified",
                template="plotly_white",
                margin=dict(t=60, b=40, l=40, r=40)
            )
            
            # On force l'axe X en mode "catégorie" si on compare des politiques pour éviter que Plotly fasse une ligne continue
            type_x = 'category' if x_col in ['Policy', 'Seed'] else '-'
            fig.update_xaxes(title_text=x_title, type=type_x, row=1, col=1)
            fig.update_xaxes(title_text=x_title, type=type_x, row=1, col=2)
            
            fig.update_yaxes(title_text="Temps (µs)", type="log", row=1, col=1)
            fig.update_yaxes(title_text="Score", row=1, col=2)

            fig.show()

    # 3. Lier les événements
    dropdown_agents.observe(update_plot, names='value')
    dropdown_objets.observe(update_plot, names='value')
    dropdown_seed.observe(update_plot, names='value')
    dropdown_policy.observe(update_plot, names='value') # NOUVEAU
    
    # 4. Affichage
    display(ui, out)
    update_plot()

In [ ]:
def plot_impact_ratio(df):
    if df.empty:
        print("Aucune donnée trouvée dans le ZIP.")
        return

    # On sépare les graphiques par nombre d'agents pour que ce soit lisible
    configurations_agents = df['Agents'].unique()
    configurations_agents.sort()

    for agents in configurations_agents:
        df_filtre = df[df['Agents'] == agents]
        
        # On groupe par Ratio pour voir son impact global (moyenne sur tous les objets et seeds)
        df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

        if len(df_grouped) < 2:
            print(f"⚠️ Pas assez de variations de 'ratioRandom' pour tracer une courbe pour {agents} agents.")
            continue

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        fig.suptitle(f"Impact du Ratio Random ({agents} Agents)", fontsize=16, fontweight='bold')

        # --- Courbe 1 : Temps d'exécution (Axe X = Ratio) ---
        # Le solveur est tracé pour vérifier si le ratio impacte la génération du problème
        ax1.plot(df_grouped['Ratio'], df_grouped['Solveur_Temps_us'], marker='o', label='Solveur', color='red', linewidth=2)
        ax1.plot(df_grouped['Ratio'], df_grouped['MCTS_Temps_Moyen_us'], marker='s', label='MCTS (Moyenne)', color='blue', linewidth=2)
        ax1.set_title("Évolution du Temps selon le Ratio")
        ax1.set_xlabel("Ratio Random")
        ax1.set_ylabel("Temps (microsecondes)")
        ax1.set_yscale('log')
        ax1.legend()
        ax1.grid(True, which="both", ls="--")

        # --- Courbe 2 : Score (Axe X = Ratio) ---
        ax2.plot(df_grouped['Ratio'], df_grouped['Solveur_Score'], marker='o', label='Solveur (Optimum)', color='red', linewidth=2)
        ax2.plot(df_grouped['Ratio'], df_grouped['MCTS_Score_Moyen'], marker='s', label='MCTS (Moyenne)', color='blue', linewidth=2)
        ax2.set_title("Évolution des Scores selon le Ratio")
        ax2.set_xlabel("Ratio Random")
        ax2.set_ylabel("Score")
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()
        plt.show()

In [ ]:
def plot_impact_ratio_interactive(df, show_all=True):
    if df.empty:
        print("Aucune donnée trouvée.")
        return
    if (show_all):
        configurations_agents = df['Agents'].unique()
        configurations_agents.sort()

        for agents in configurations_agents:
            df_filtre = df[df['Agents'] == agents]
            
            # On groupe par Ratio
            df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

            if len(df_grouped) < 2:
                continue

            # Création de la figure avec deux colonnes
            fig = make_subplots(
                rows=1, cols=2, 
                subplot_titles=("Temps selon le Ratio", "Score selon le Ratio")
            )

            # --- Graphique 1 : Temps ---
            fig.add_trace(go.Scatter(
                x=df_grouped['Ratio'], y=df_grouped['Solveur_Temps_us'],
                mode='lines+markers', name='Solveur (Temps)', line=dict(color='red')
            ), row=1, col=1)
            
            fig.add_trace(go.Scatter(
                x=df_grouped['Ratio'], y=df_grouped['MCTS_Temps_Moyen_us'],
                mode='lines+markers', name='MCTS (Temps)', line=dict(color='blue')
            ), row=1, col=1)

            # --- Graphique 2 : Score ---
            fig.add_trace(go.Scatter(
                x=df_grouped['Ratio'], y=df_grouped['Solveur_Score'],
                mode='lines+markers', name='Solveur (Score)', line=dict(color='red', dash='dot')
            ), row=1, col=2)
            
            fig.add_trace(go.Scatter(
                x=df_grouped['Ratio'], y=df_grouped['MCTS_Score_Moyen'],
                mode='lines+markers', name='MCTS (Score)', line=dict(color='blue', dash='dot')
            ), row=1, col=2)

            # --- Mise en forme ---
            fig.update_layout(
                title_text=f"Impact du Ratio Random pour {agents} Agents",
                title_font=dict(size=20),
                hovermode="x unified", # Le tooltip global très lisible
                template="plotly_white"
            )
            
            # Axe X
            fig.update_xaxes(title_text="Ratio Random", row=1, col=1)
            fig.update_xaxes(title_text="Ratio Random", row=1, col=2)
            
            # Axe Y (Log pour le temps)
            fig.update_yaxes(title_text="Temps (µs)", type="log", row=1, col=1)
            fig.update_yaxes(title_text="Score", row=1, col=2)

            fig.show()
    else:
        
        # 1. Options des widgets
        agents_options = ['Tous'] + sorted(df['Agents'].unique().tolist())
        objets_options = ['Tous'] + sorted(df['Objets'].unique().tolist())
        
        if 'Seed' in df.columns:
            seed_options = ['Tous'] + sorted(df['Seed'].unique().tolist())
        else:
            seed_options = ["Non défini"]

        # 2. Création des widgets
        dropdown_agents = widgets.Dropdown(
            options=agents_options, 
            value=agents_options[1] if len(agents_options)>1 else 'Tous', 
            description='👥 Agents:', 
            layout={'width': 'max-content'}
        )
        dropdown_objets = widgets.Dropdown(
            options=objets_options, 
            value='Tous', 
            description='📦 Objets:', 
            layout={'width': 'max-content'}
        )
        dropdown_seed = widgets.Dropdown(
            options=seed_options, 
            value='Tous', 
            description='🌱 Graine:', 
            layout={'width': 'max-content'}
        )
        
        # Zone d'affichage
        ui = widgets.HBox([dropdown_agents, dropdown_objets, dropdown_seed])
        out = widgets.Output()

        def update_plot(change=None):
            with out:
                out.clear_output(wait=True)
                
                ag = dropdown_agents.value
                ob = dropdown_objets.value
                sd = dropdown_seed.value
                
                # Filtrage dynamique
                df_filtre = df.copy()
                if ag != 'Tous':
                    df_filtre = df_filtre[df_filtre['Agents'] == ag]
                if ob != 'Tous':
                    df_filtre = df_filtre[df_filtre['Objets'] == ob]
                if sd != 'Tous' and sd != "Non défini":
                    df_filtre = df_filtre[df_filtre['Seed'] == sd]

                if df_filtre.empty:
                    print(f"⚠️ Aucune donnée trouvée pour Agents: {ag}, Objets: {ob}, Graine: {sd}.")
                    return

                if 'Ratio' not in df_filtre.columns:
                    print("⚠️ La colonne 'Ratio' n'existe pas dans ce DataFrame.")
                    return

                # On groupe par Ratio pour voir son impact global
                df_grouped = df_filtre.groupby('Ratio').mean().reset_index()
                df_grouped = df_grouped.sort_values('Ratio')

                if len(df_grouped) < 2:
                    print(f"⚠️ Pas assez de variations de 'Ratio' pour tracer une courbe avec Agents: {ag}, Objets: {ob}, Graine: {sd}.")
                    return

                # Création de la figure avec deux colonnes
                fig = make_subplots(
                    rows=1, cols=2, 
                    subplot_titles=("Temps selon le Ratio", "Score selon le Ratio")
                )

                # --- Graphique 1 : Temps ---
                fig.add_trace(go.Scatter(
                    x=df_grouped['Ratio'], y=df_grouped['Solveur_Temps_us'],
                    mode='lines+markers', name='Solveur (Temps)', line=dict(color='red')
                ), row=1, col=1)
                
                fig.add_trace(go.Scatter(
                    x=df_grouped['Ratio'], y=df_grouped['MCTS_Temps_Moyen_us'],
                    mode='lines+markers', name='MCTS Moyenne (Temps)', line=dict(color='blue')
                ), row=1, col=1)

                # --- Graphique 2 : Scores Absolus ---
                fig.add_trace(go.Scatter(
                    x=df_grouped['Ratio'], y=df_grouped['Solveur_Score'],
                    mode='lines+markers', name='Solveur (Score)', line=dict(color='red', dash='dot')
                ), row=1, col=2)
                
                fig.add_trace(go.Scatter(
                    x=df_grouped['Ratio'], y=df_grouped['MCTS_Score_Moyen'],
                    mode='lines+markers', name='MCTS Moyenne (Score)', line=dict(color='blue', dash='dot')
                ), row=1, col=2)

                # --- Mise en forme ---
                titre = f"Impact du Ratio Random — Agents: {ag} | Objets: {ob} | Graine: {sd}"
                fig.update_layout(
                    title_text=titre,
                    title_font=dict(size=18),
                    hovermode="x unified", # Tooltip global très lisible
                    template="plotly_white",
                    margin=dict(t=60, b=40, l=40, r=40)
                )
                
                # Axe X
                fig.update_xaxes(title_text="Ratio Random", row=1, col=1)
                fig.update_xaxes(title_text="Ratio Random", row=1, col=2)
                
                # Axe Y (Log pour le temps)
                fig.update_yaxes(title_text="Temps (µs)", type="log", row=1, col=1)
                fig.update_yaxes(title_text="Score", row=1, col=2)

                fig.show()   
        dropdown_agents.observe(update_plot, names='value')
        dropdown_objets.observe(update_plot, names='value')
        dropdown_seed.observe(update_plot, names='value')
        
        # 4. Afficher le menu et lancer le premier rendu
        display(ui, out)
        update_plot()

In [ ]:
def plot_ratio_impact_on_score(df, combine_agents=True):
    if df.empty:
        print("Aucune donnée trouvée dans le ZIP.")
        return

    # On copie le DataFrame et on s'assure de ne pas diviser par zéro
    df_score = df[df['Solveur_Score'] > 0].copy()
    
    # Calcul du pourcentage d'optimalité
    df_score['Optimalite_MCTS_%'] = (df_score['MCTS_Score_Moyen'] / df_score['Solveur_Score']) * 100

    configurations_agents = df_score['Agents'].unique()
    configurations_agents.sort()

    # --- CAS 1 : Tout sur le même graphique ---
    if combine_agents:
        plt.figure(figsize=(8, 5)) 

        for agents in configurations_agents:
            df_filtre = df_score[df_score['Agents'] == agents]
            df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

            if len(df_grouped) < 2:
                print(f"⚠️ Pas assez de variations de 'ratioRandom' pour {agents} agents.")
                continue
            
            plt.plot(df_grouped['Ratio'], df_grouped['Optimalite_MCTS_%'], 
                     marker='o', linewidth=2, label=f'MCTS ({agents} Agents)')

        plt.axhline(y=100, color='red', linestyle='--', linewidth=2, label='Optimum (Solveur = 100%)')
        plt.title("Impact du Ratio sur la qualité du score (Toutes configurations)", fontsize=14, fontweight='bold')
        plt.xlabel("Ratio Random")
        plt.ylabel("Qualité de la solution (%)")
        plt.ylim(0, 105) 
        plt.legend()
        plt.grid(True, which="both", ls="--")
        plt.tight_layout()
        plt.show()

    # --- CAS 2 : Un graphique séparé par nombre d'agents ---
    else:
        for agents in configurations_agents:
            df_filtre = df_score[df_score['Agents'] == agents]
            df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

            if len(df_grouped) < 2:
                print(f"⚠️ Pas assez de variations de 'ratioRandom' pour {agents} agents.")
                continue

            plt.figure(figsize=(7, 4))
            
            plt.plot(df_grouped['Ratio'], df_grouped['Optimalite_MCTS_%'], 
                     marker='o', color='purple', linewidth=2, label='Score MCTS')
            plt.axhline(y=100, color='red', linestyle='--', linewidth=2, label='Optimum (Solveur = 100%)')
            
            plt.title(f"Impact du Ratio sur la qualité du score ({agents} Agents)", fontsize=14, fontweight='bold')
            plt.xlabel("Ratio Random")
            plt.ylabel("Qualité de la solution (%)")
            plt.ylim(0, 105) 
            plt.legend()
            plt.grid(True, which="both", ls="--")
            plt.tight_layout()
            plt.show()

In [ ]:
def plot_ratio_impact_on_score_interactive(df, combine_agents=True, show_all=True):
    if df.empty:
        print("Aucune donnée trouvée.")
        return
    if (show_all):
        df_score = df[df['Solveur_Score'] > 0].copy()
        df_score['Optimalite_MCTS_%'] = (df_score['MCTS_Score_Moyen'] / df_score['Solveur_Score']) * 100
        df_score['Optimalite_MCTS_%'] = df_score['Optimalite_MCTS_%'].clip(upper=100)

        # --- CHANGEMENT 1 : Calcul du Min, Max et de la Moyenne ---
        df_grouped = df_score.groupby(['Agents', 'Ratio']).agg(
            Optimalite_Moyenne=('Optimalite_MCTS_%', 'mean'),
            Optimalite_Min=('Optimalite_MCTS_%', 'min'),
            Optimalite_Max=('Optimalite_MCTS_%', 'max'),
            Score_MCTS_Moyen=('MCTS_Score_Moyen', 'mean'),
            Score_Solveur=('Solveur_Score', 'mean')
        ).reset_index()
        
        df_grouped['Configuration'] = df_grouped['Agents'].astype(str) + " Agents"

        if combine_agents:
            y_min = max(0, df_grouped['Optimalite_Min'].min() )
            y_max = 100
            
            fig = px.line(
                df_grouped,
                x="Ratio",
                y="Optimalite_Moyenne", # On pointe vers la nouvelle colonne
                color="Configuration",
                markers=True,
                title="Impact du Ratio sur la qualité du score (Toutes configurations)",
                labels={
                    "Ratio": "Ratio Random",
                    "Optimalite_Moyenne": "Qualité de la solution (%)"
                },
                # --- CHANGEMENT 2 : Ajout du Min et Max dans l'infobulle ---
                hover_data={
                    "Configuration": False, 
                    "Ratio": ':.2f',
                    "Optimalite_Moyenne": ':.2f', 
                    "Optimalite_Min": ':.2f',  # <- Ajout
                    "Optimalite_Max": ':.2f',  # <- Ajout
                    "Score_MCTS_Moyen": ':.4f',
                    "Score_Solveur": ':.4f'
                }
            )
            
            fig.add_hline(y=100, line_dash="dash", line_color="red", annotation_text="Optimum (100%)")
            fig.update_layout(yaxis=dict(range=[y_min, y_max]), hovermode="x unified", template="plotly_white")
            fig.show()

        else:
            configurations_agents = df_grouped['Agents'].unique()
            configurations_agents.sort()
            
            for agents in configurations_agents:
                df_filtre = df_grouped[df_grouped['Agents'] == agents]
                
                if len(df_filtre) < 2:
                    continue

                y_min_local = max(0, df_filtre['Optimalite_Min'].min() - 2)
                y_max_local = 102
                
                fig = px.line(
                    df_filtre,
                    x="Ratio",
                    y="Optimalite_Moyenne", # On pointe vers la nouvelle colonne
                    markers=True,
                    title=f"Impact du Ratio sur la qualité du score ({agents} Agents)",
                    labels={
                        "Ratio": "Ratio Random",
                        "Optimalite_Moyenne": "Qualité de la solution (%)"
                    },
                    color_discrete_sequence=['purple'],
                    # --- CHANGEMENT 2 : Ajout du Min et Max dans l'infobulle ---
                    hover_data={
                        "Ratio": ':.2f',
                        "Optimalite_Moyenne": ':.2f', 
                        "Optimalite_Min": ':.2f',  # <- Ajout
                        "Optimalite_Max": ':.2f',  # <- Ajout
                        "Score_MCTS_Moyen": ':.4f',
                        "Score_Solveur": ':.4f'
                    }
                )
                
                fig.add_hline(y=100, line_dash="dash", line_color="red", annotation_text="Optimum (100%)")
                fig.update_layout(yaxis=dict(range=[y_min_local, y_max_local]), hovermode="x unified", template="plotly_white")
                fig.show()
    else:
        # 1. Normalisation préalable et sécurisée
        df_score = df[df['Solveur_Score'] > 0].copy()
        df_score['Optimalite_MCTS_%'] = (df_score['MCTS_Score_Moyen'] / df_score['Solveur_Score']) * 100
        df_score['Optimalite_MCTS_%'] = df_score['Optimalite_MCTS_%'].clip(upper=100)

        # 2. Options des widgets
        agents_options = ['Tous'] + sorted(df_score['Agents'].unique().tolist())
        objets_options = ['Tous'] + sorted(df_score['Objets'].unique().tolist())
        
        if 'Seed' in df_score.columns:
            seed_options = ['Tous'] + sorted(df_score['Seed'].unique().tolist())
        else:
            seed_options = ["Non défini"]

        # 3. Création des widgets
        dropdown_agents = widgets.Dropdown(
            options=agents_options, 
            value='Tous', # Par défaut sur "Tous" pour voir toutes les courbes d'un coup
            description='👥 Agents:', 
            layout={'width': 'max-content'}
        )
        dropdown_objets = widgets.Dropdown(
            options=objets_options, 
            value='Tous', 
            description='📦 Objets:', 
            layout={'width': 'max-content'}
        )
        dropdown_seed = widgets.Dropdown(
            options=seed_options, 
            value='Tous', 
            description='🌱 Graine:', 
            layout={'width': 'max-content'}
        )

        # Zone d'affichage
        ui = widgets.HBox([dropdown_agents, dropdown_objets, dropdown_seed])
        out = widgets.Output()

        def update_plot(change=None):
            with out:
                out.clear_output(wait=True)
                
                ag = dropdown_agents.value
                ob = dropdown_objets.value
                sd = dropdown_seed.value

                # Filtrage selon les objets et la graine (On garde les agents pour le 'groupby')
                df_filtre = df_score.copy()
                if ob != 'Tous':
                    df_filtre = df_filtre[df_filtre['Objets'] == ob]
                if sd != 'Tous' and sd != "Non défini":
                    df_filtre = df_filtre[df_filtre['Seed'] == sd]
                if ag != 'Tous':
                    df_filtre = df_filtre[df_filtre['Agents'] == ag]

                if df_filtre.empty:
                    print(f"⚠️ Aucune donnée trouvée pour Agents: {ag}, Objets: {ob}, Graine: {sd}.")
                    return
                    
                if 'Ratio' not in df_filtre.columns or len(df_filtre['Ratio'].unique()) < 2:
                    print(f"⚠️ Pas assez de variations de 'Ratio' pour cette configuration.")
                    return

                # --- CALCUL DES AGGRÉGATIONS ---
                # On groupe toujours par Ratio et Agents (pour différencier les courbes si ag == 'Tous')
                df_grouped = df_filtre.groupby(['Agents', 'Ratio']).agg(
                    Optimalite_Moyenne=('Optimalite_MCTS_%', 'mean'),
                    Optimalite_Min=('Optimalite_MCTS_%', 'min'),
                    Optimalite_Max=('Optimalite_MCTS_%', 'max'),
                    Score_MCTS_Moyen=('MCTS_Score_Moyen', 'mean'),
                    Score_Solveur=('Solveur_Score', 'mean')
                ).reset_index()
                
                df_grouped['Configuration'] = df_grouped['Agents'].astype(str) + " Agents"
                df_grouped = df_grouped.sort_values('Ratio')

                # --- CRÉATION DU GRAPHIQUE ---
                y_min = max(0, df_grouped['Optimalite_Min'].min() - 2)
                y_max = 102
                
                titre = f"Impact du Ratio sur l'Optimalité — Agents: {ag} | Objets: {ob} | Graine: {sd}"

                fig = px.line(
                    df_grouped,
                    x="Ratio",
                    y="Optimalite_Moyenne",
                    # Si 'Tous' est sélectionné, on met une couleur par Agent. Sinon, ligne unique mauve par défaut.
                    color="Configuration" if ag == 'Tous' else None, 
                    color_discrete_sequence=None if ag == 'Tous' else ['purple'],
                    markers=True,
                    title=titre,
                    labels={
                        "Ratio": "Ratio Random",
                        "Optimalite_Moyenne": "Qualité de la solution (%)"
                    },
                    hover_data={
                        "Configuration": False, # Caché de la bulle car déjà indiqué par la couleur/légende
                        "Ratio": ':.2f',
                        "Optimalite_Moyenne": ':.2f', 
                        "Optimalite_Min": ':.2f',  
                        "Optimalite_Max": ':.2f',  
                        "Score_MCTS_Moyen": ':.4f',
                        "Score_Solveur": ':.4f'
                    }
                )
                
                # Ligne de l'optimum 100%
                fig.add_hline(y=100, line_dash="dash", line_color="red", annotation_text="Optimum (100%)")
                
                # Mise en page
                fig.update_layout(
                    yaxis=dict(range=[y_min, y_max]), 
                    hovermode="x unified", 
                    template="plotly_white",
                    margin=dict(t=60, b=40, l=40, r=40)
                )
                
                fig.show()

        # 4. Lier l'événement de changement des 3 widgets au graphique
        dropdown_agents.observe(update_plot, names='value')
        dropdown_objets.observe(update_plot, names='value')
        dropdown_seed.observe(update_plot, names='value')
        
        # 5. Afficher le menu et lancer le premier rendu
        display(ui, out)
        update_plot()

In [ ]:
def plot_erreur_normalisee(df, combine_agents=True):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    # On copie le DataFrame et on s'assure de ne pas diviser par zéro
    df_safe = df[df['Solveur_Score'] > 0].copy()

    # Calcul de l'erreur normalisée globale en pourcentage pour chaque ligne
    df_safe['Erreur_%'] = ((df_safe['Solveur_Score'] - df_safe['MCTS_Score_Moyen']) / df_safe['Solveur_Score']) * 100

    configurations_agents = df_safe['Agents'].unique()
    configurations_agents.sort()

    # --- CAS 1 : Toutes les courbes sur le même graphique ---
    if combine_agents:
        plt.figure(figsize=(10, 6))

        for agents in configurations_agents:
            df_filtre = df_safe[df_safe['Agents'] == agents]
            df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

            if len(df_grouped) < 2:
                continue
                
            plt.plot(df_grouped['Ratio'], df_grouped['Erreur_%'], 
                     marker='o', linewidth=2, label=f'MCTS ({agents} Agents)')

        plt.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Optimum (Erreur = 0%)')

        plt.title("Erreur Normalisée globale du MCTS selon le Ratio", fontsize=14, fontweight='bold')
        plt.xlabel("Ratio Random")
        plt.ylabel("Erreur par rapport au Solveur (%)")
        
        erreur_max = df_safe['Erreur_%'].max()
        plt.ylim(-1, erreur_max + 2 if pd.notna(erreur_max) else 10)
        
        plt.grid(True, which="both", ls="--")
        plt.legend()
        plt.tight_layout()
        plt.show()

    # --- CAS 2 : Un graphique séparé par configuration d'agents ---
    else:
        for agents in configurations_agents:
            df_filtre = df_safe[df_safe['Agents'] == agents]
            df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

            if len(df_grouped) < 2:
                continue
            
            plt.figure(figsize=(7, 4))
            
            # Utilisation d'une couleur neutre si c'est affiché seul
            plt.plot(df_grouped['Ratio'], df_grouped['Erreur_%'], 
                     marker='o', color='purple', linewidth=2, label='Erreur MCTS')

            plt.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Optimum (Erreur = 0%)')

            plt.title(f"Erreur Normalisée du MCTS selon le Ratio ({agents} Agents)", fontsize=14, fontweight='bold')
            plt.xlabel("Ratio Random")
            plt.ylabel("Erreur par rapport au Solveur (%)")
            
            # Ajustement dynamique du zoom pour ce graphique spécifique
            erreur_max = df_filtre['Erreur_%'].max()
            plt.ylim(-1, erreur_max + 2 if pd.notna(erreur_max) else 10)
            
            plt.grid(True, which="both", ls="--")
            plt.legend()
            plt.tight_layout()
            plt.show()
         

In [ ]:
def plot_erreur_normalisee_interactive(df, combine_agents=True, show_all=True):
    if df.empty:
        print("Aucune donnée trouvée.")
        return
    if(show_all):
        # Préparation des données
        df_safe = df[df['Solveur_Score'] > 0].copy()
        df_safe['Erreur_%'] = ((df_safe['Solveur_Score'] - df_safe['MCTS_Score_Moyen']) / df_safe['Solveur_Score']) * 100

        # On groupe par Agents et Ratio pour faire les moyennes
        df_grouped = df_safe.groupby(['Agents', 'Ratio']).mean().reset_index()
        
        # On crée une colonne texte pour que Plotly gère bien les catégories/couleurs
        df_grouped['Configuration'] = df_grouped['Agents'].astype(str) + " Agents"

        if combine_agents:
            # --- Création du graphique interactif (Toutes les courbes) ---
            fig = px.line(
                df_grouped,
                x="Ratio",
                y="Erreur_%",
                color="Configuration",
                markers=True,
                title="Erreur Normalisée globale du MCTS selon le Ratio",
                labels={
                    "Ratio": "Ratio Random",
                    "Erreur_%": "Erreur par rapport au Solveur (%)"
                },
                # Configuration de la bulle d'info au survol (hover)
                hover_data={
                    "Configuration": False, # On le cache de la liste car c'est le titre de la bulle
                    "Ratio": ':.2f',
                    "Erreur_%": ':.2f', 
                    "MCTS_Score_Moyen": ':.2f',
                    "Solveur_Score": ':.2f'
                }
            )
            
            # Ajout de la ligne rouge de l'optimum
            fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Optimum (0%)")
            
            # Forcer l'axe Y à descendre un peu sous le 0
            fig.update_layout(yaxis=dict(range=[-1, df_grouped['Erreur_%'].max() + 2]))
            
            # Ouvre le graphique dans ton navigateur web
            fig.show()

        else:
            # --- Création de graphiques séparés ---
            configurations_agents = df_grouped['Agents'].unique()
            configurations_agents.sort()
            
            for agents in configurations_agents:
                df_filtre = df_grouped[df_grouped['Agents'] == agents]
                
                fig = px.line(
                    df_filtre,
                    x="Ratio",
                    y="Erreur_%",
                    markers=True,
                    title=f"Erreur Normalisée du MCTS selon le Ratio ({agents} Agents)",
                    labels={
                        "Ratio": "Ratio Random",
                        "Erreur_%": "Erreur par rapport au Solveur (%)"
                    },
                    color_discrete_sequence=['purple'],
                    hover_data={
                        "Ratio": ':.2f',
                        "Erreur_%": ':.2f', 
                        "MCTS_Score_Moyen": ':.2f',
                        "Solveur_Score": ':.2f'
                    }
                )
                
                fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Optimum (0%)")
                fig.update_layout(yaxis=dict(range=[-1, df_filtre['Erreur_%'].max() + 2]))
                fig.show()
    else: 
        # 1. Normalisation préalable et sécurisée
        df_safe = df[df['Solveur_Score'] > 0].copy()
        # Calcul de l'erreur en pourcentage : ((Solveur - MCTS) / Solveur) * 100
        df_safe['Erreur_%'] = ((df_safe['Solveur_Score'] - df_safe['MCTS_Score_Moyen']) / df_safe['Solveur_Score']) * 100

        # 2. Options des widgets
        agents_options = ['Tous'] + sorted(df_safe['Agents'].unique().tolist())
        objets_options = ['Tous'] + sorted(df_safe['Objets'].unique().tolist())
        
        if 'Seed' in df_safe.columns:
            seed_options = ['Tous'] + sorted(df_safe['Seed'].unique().tolist())
        else:
            seed_options = ["Non défini"]

        # 3. Création des widgets
        dropdown_agents = widgets.Dropdown(
            options=agents_options, 
            value='Tous', # Par défaut sur "Tous" pour voir toutes les courbes d'un coup
            description='👥 Agents:', 
            layout={'width': 'max-content'}
        )
        dropdown_objets = widgets.Dropdown(
            options=objets_options, 
            value='Tous', 
            description='📦 Objets:', 
            layout={'width': 'max-content'}
        )
        dropdown_seed = widgets.Dropdown(
            options=seed_options, 
            value='Tous', 
            description='🌱 Graine:', 
            layout={'width': 'max-content'}
        )

        # Zone d'affichage
        ui = widgets.HBox([dropdown_agents, dropdown_objets, dropdown_seed])
        out = widgets.Output()

        def update_plot(change=None):
            with out:
                out.clear_output(wait=True)
                
                ag = dropdown_agents.value
                ob = dropdown_objets.value
                sd = dropdown_seed.value

                # Filtrage selon les objets et la graine (On garde les agents pour le 'groupby')
                df_filtre = df_safe.copy()
                if ob != 'Tous':
                    df_filtre = df_filtre[df_filtre['Objets'] == ob]
                if sd != 'Tous' and sd != "Non défini":
                    df_filtre = df_filtre[df_filtre['Seed'] == sd]
                if ag != 'Tous':
                    df_filtre = df_filtre[df_filtre['Agents'] == ag]

                if df_filtre.empty:
                    print(f"⚠️ Aucune donnée trouvée pour Agents: {ag}, Objets: {ob}, Graine: {sd}.")
                    return
                    
                if 'Ratio' not in df_filtre.columns or len(df_filtre['Ratio'].unique()) < 2:
                    print(f"⚠️ Pas assez de variations de 'Ratio' pour cette configuration.")
                    return

                # --- CALCUL DES AGGRÉGATIONS ---
                df_grouped = df_filtre.groupby(['Agents', 'Ratio']).agg(
                    Erreur_Moyenne=('Erreur_%', 'mean'),
                    Erreur_Min=('Erreur_%', 'min'),
                    Erreur_Max=('Erreur_%', 'max'),
                    Score_MCTS_Moyen=('MCTS_Score_Moyen', 'mean'),
                    Score_Solveur=('Solveur_Score', 'mean')
                ).reset_index()
                
                df_grouped['Configuration'] = df_grouped['Agents'].astype(str) + " Agents"
                df_grouped = df_grouped.sort_values('Ratio')

                # --- CRÉATION DU GRAPHIQUE ---
                y_min = -1 # On descend un tout petit peu sous zéro pour voir la ligne d'optimum
                # S'il n'y a pas d'erreur max valide, on met une limite par défaut à 10%
                max_err = df_grouped['Erreur_Max'].max()
                y_max = max_err + 2 if pd.notna(max_err) else 10
                
                titre = f"Impact du Ratio sur l'Erreur Normalisée — Agents: {ag} | Objets: {ob} | Graine: {sd}"

                fig = px.line(
                    df_grouped,
                    x="Ratio",
                    y="Erreur_Moyenne",
                    # Si 'Tous' est sélectionné, on met une couleur par Agent. Sinon, ligne unique rouge.
                    color="Configuration" if ag == 'Tous' else None, 
                    color_discrete_sequence=None if ag == 'Tous' else ['red'],
                    markers=True,
                    title=titre,
                    labels={
                        "Ratio": "Ratio Random",
                        "Erreur_Moyenne": "Erreur par rapport au Solveur (%)"
                    },
                    hover_data={
                        "Configuration": False, 
                        "Ratio": ':.2f',
                        "Erreur_Moyenne": ':.2f', 
                        "Erreur_Min": ':.2f',  
                        "Erreur_Max": ':.2f',  
                        "Score_MCTS_Moyen": ':.4f',
                        "Score_Solveur": ':.4f'
                    }
                )
                
                # Ligne de l'optimum 0% d'erreur (en vert cette fois !)
                fig.add_hline(y=0, line_dash="dash", line_color="green", annotation_text="Optimum (0% d'erreur)")
                
                # Mise en page
                fig.update_layout(
                    yaxis=dict(range=[y_min, y_max]), 
                    hovermode="x unified", 
                    template="plotly_white",
                    margin=dict(t=60, b=40, l=40, r=40)
                )
                
                fig.show()

        # 4. Lier l'événement de changement des 3 widgets au graphique
        dropdown_agents.observe(update_plot, names='value')
        dropdown_objets.observe(update_plot, names='value')
        dropdown_seed.observe(update_plot, names='value')
        
        # 5. Afficher le menu et lancer le premier rendu
        display(ui, out)
        update_plot()

In [ ]:
def plot_metrics_comparison_interactive(df):
    if df.empty:
        print("Aucune donnée trouvée.")
        return
        
    df_base = df.copy()
            
    # 1. Options des widgets (basé sur toutes les données)
    agents_options = ['Tous'] + sorted(df_base['Agents'].unique().tolist())
    objets_options = ['Tous'] + sorted(df_base['Objets'].unique().tolist())
    
    if 'Ratio' in df_base.columns:
        ratio_options = ['Tous'] + sorted(df_base['Ratio'].unique().tolist())
    else:
        ratio_options = ["Non défini"]
        
    if 'Seed' in df_base.columns:
        seed_options = ['Tous'] + sorted(df_base['Seed'].unique().tolist())
    else:
        seed_options = ["Non défini"]

    # 2. Création des widgets
    dropdown_agents = widgets.Dropdown(options=agents_options, value='Tous', description='👥 Agents:', layout={'width': 'max-content'})
    dropdown_objets = widgets.Dropdown(options=objets_options, value='Tous', description='📦 Objets:', layout={'width': 'max-content'})
    dropdown_seed = widgets.Dropdown(options=seed_options, value='Tous', description='🌱 Graine:', layout={'width': 'max-content'})
    
    slider_ratio = widgets.SelectionSlider(
        options=ratio_options,
        value=ratio_options[1] if len(ratio_options)>1 else 'Tous',
        description='🎲 Ratio:',
        disabled=False,
        continuous_update=False, 
        orientation='horizontal',
        readout=True,
        layout={'width': '400px'}
    )
    
    # NOUVEAU : Widget Checkbox pour le filtre des zéros
    checkbox_filter_zeros = widgets.Checkbox(
        value=True, # Coché par défaut
        description="🚫 Exclure les scores du solveur = 0",
        indent=False,
        layout={'width': 'max-content', 'margin': '0 0 0 20px'}
    )

    # Organisation de l'interface : 3 menus + Checkbox en haut, le slider en dessous
    ui_top = widgets.HBox([dropdown_agents, dropdown_objets, dropdown_seed, checkbox_filter_zeros])
    ui = widgets.VBox([ui_top, slider_ratio])
    
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            
            ag = dropdown_agents.value
            ob = dropdown_objets.value
            rat = slider_ratio.value
            sd = dropdown_seed.value
            filtrer_zeros = checkbox_filter_zeros.value

            # 3. Filtrage dynamique
            df_filtre = df_base.copy()
            
            # Application du filtre des zéros SI la case est cochée
            if filtrer_zeros and 'Solveur_Score' in df_filtre.columns:
                df_filtre = df_filtre[df_filtre['Solveur_Score'] != 0]
                
            if ag != 'Tous':
                df_filtre = df_filtre[df_filtre['Agents'] == ag]
            if ob != 'Tous':
                df_filtre = df_filtre[df_filtre['Objets'] == ob]
            if rat != 'Tous' and rat != "Non défini":
                df_filtre = df_filtre[df_filtre['Ratio'] == rat]
            if sd != 'Tous' and sd != "Non défini":
                df_filtre = df_filtre[df_filtre['Seed'] == sd]

            if df_filtre.empty:
                print(f"⚠️ Aucune donnée trouvée pour cette sélection.")
                return

            # Vérification des colonnes de métriques disponibles
            metrics_list = ["ParetoOptimal", "Prop", "EF", "EFX", "EF1"]
            metrics_list = [m for m in metrics_list if f"Solveur_{m}" in df_filtre.columns]

            if not metrics_list:
                print("⚠️ Aucune colonne de métrique trouvée dans les données.")
                return

            # 4. Calcul des moyennes sur les données filtrées
            means = []
            for m in metrics_list:
                solveur_val = df_filtre[f"Solveur_{m}"].mean() * 100
                mcts_val = df_filtre[f"MCTS_{m}"].mean() * 100
                means.append({"Metrique": m, "Solveur (%)": solveur_val, "MCTS (%)": mcts_val})
                
            df_means = pd.DataFrame(means)

            # 5. Création du graphique à barres
            fig = go.Figure()
            fig.add_trace(go.Bar(
                x=df_means['Metrique'], y=df_means['Solveur (%)'],
                name='Solveur', marker_color='red'
            ))
            fig.add_trace(go.Bar(
                x=df_means['Metrique'], y=df_means['MCTS (%)'],
                name='MCTS (Moyenne)', marker_color='blue'
            ))

            # Construction du titre
            titre = f"Respect des Métriques — Agents: {ag} | Objets: {ob} | Ratio: {rat} | Graine: {sd}"
            if filtrer_zeros:
                titre += " (Scores > 0)"

            fig.update_layout(
                title_text=titre,
                title_font=dict(size=18),
                xaxis_title="Métriques",
                yaxis_title="Taux de réussite (%)",
                barmode='group',
                template="plotly_white",
                yaxis=dict(range=[0, 105]),
                margin=dict(t=60, b=40, l=40, r=40)
            )
            fig.show()

    # 6. Lier les événements de tous les widgets
    dropdown_agents.observe(update_plot, names='value')
    dropdown_objets.observe(update_plot, names='value')
    slider_ratio.observe(update_plot, names='value')
    dropdown_seed.observe(update_plot, names='value')
    checkbox_filter_zeros.observe(update_plot, names='value') # On lie la case à cocher !
    
    # 7. Affichage final
    display(ui, out)
    update_plot()

In [ ]:
def widget_variance_mcts_normalise(df):
    """
    Crée un widget interactif affichant la variance du MCTS en POURCENTAGE 
    du score optimal, permettant de comparer équitablement différentes graines.
    """
    if df.empty:
        print("Le DataFrame est vide.")
        return
        
    # Sécurité : on écarte les instances où le score du solveur est 0 pour éviter la division par zéro
    df_safe = df[df['Solveur_Score'] > 0].copy()
    
    # 1. Identifier les valeurs uniques
    agents_options = ['Tous'] + sorted(df_safe['Agents'].unique().tolist())
    objets_options = ['Tous'] + sorted(df_safe['Objets'].unique().tolist())
    
    ratio_options = ['Tous'] + sorted(df_safe['Ratio'].unique().tolist()) if 'Ratio' in df_safe.columns else ["Non défini"]
    seed_options = ['Tous'] + sorted(df_safe['Seed'].unique().tolist()) if 'Seed' in df_safe.columns else ["Non défini"]
        
    # 2. Création des widgets
    dropdown_agents = widgets.Dropdown(options=agents_options, value=agents_options[1] if len(agents_options)>1 else 'Tous', description='👥 Agents:', layout={'width': 'max-content'})
    dropdown_objets = widgets.Dropdown(options=objets_options, value=objets_options[1] if len(objets_options)>1 else 'Tous', description='📦 Objets:', layout={'width': 'max-content'})
    dropdown_seed = widgets.Dropdown(options=seed_options, value='Tous', description='🌱 Graine:', layout={'width': 'max-content'})
    
    slider_ratio = widgets.SelectionSlider(
        options=ratio_options,
        value=ratio_options[1] if len(ratio_options)>1 else 'Tous',
        description='🎲 Ratio:',
        disabled=False,
        continuous_update=False, 
        orientation='horizontal',
        readout=True,
        layout={'width': '400px'}
    )
    
    ui_top = widgets.HBox([dropdown_agents, dropdown_objets, dropdown_seed])
    ui = widgets.VBox([ui_top, slider_ratio])
    
    out = widgets.Output()
    
    def update_plot(change=None):
        out.clear_output(wait=True)
        
        ag = dropdown_agents.value
        ob = dropdown_objets.value
        sd = dropdown_seed.value
        rat = slider_ratio.value
        
        # 3. Filtrage dynamique
        df_filtre = df_safe.copy()
        
        if ag != 'Tous': df_filtre = df_filtre[df_filtre['Agents'] == ag]
        if ob != 'Tous': df_filtre = df_filtre[df_filtre['Objets'] == ob]
        if sd != 'Tous' and sd != "Non défini": df_filtre = df_filtre[df_filtre['Seed'] == sd]
        if rat != 'Tous' and rat != "Non défini": df_filtre = df_filtre[df_filtre['Ratio'] == rat]
            
        with out:
            if df_filtre.empty:
                print(f"⚠️ Aucune instance trouvée pour cette configuration.")
                return
                
            # --- NOUVEAU : Calcul des pourcentages d'optimalité ---
            df_filtre['Opti_Moyen_%'] = (df_filtre['MCTS_Score_Moyen'] / df_filtre['Solveur_Score']) * 100
            df_filtre['Opti_Min_%'] = (df_filtre['MCTS_Score_Min'] / df_filtre['Solveur_Score']) * 100
            df_filtre['Opti_Max_%'] = (df_filtre['MCTS_Score_Max'] / df_filtre['Solveur_Score']) * 100
            
            # On trie simplement par index ou par graine pour l'affichage
            df_filtre = df_filtre.sort_values('Seed' if 'Seed' in df_filtre.columns else 'Solveur_Score').reset_index(drop=True)
            
            fig = go.Figure()
            
            # A. Ligne de l'optimum (100%)
            fig.add_hline(
                y=100, 
                line_dash="dash", 
                line_color="red", 
                annotation_text="Score Optimal (Solveur) = 100%", 
                annotation_position="bottom right"
            )
            
            # B. Points MCTS en pourcentage avec barres d'erreur
            hover_text = (
                "<b>Instance n°%{x}</b><br>"
                "Score MCTS Moyen : %{customdata[0]:.2f} (%{y:.1f}% de l'optimum)<br>"
                "Score Solveur absolu : %{customdata[1]:.2f}"
            )
            if sd == 'Tous' and 'Seed' in df_filtre.columns:
                hover_text += "<br><br>🌱 <b>Graine utilisée: %{customdata[2]}</b>"
                
            # Préparation des customdata pour l'infobulle (Score MCTS absolu, Score Solveur absolu, Seed)
            custom_data = df_filtre[['MCTS_Score_Moyen', 'Solveur_Score', 'Seed']].values if 'Seed' in df_filtre.columns else df_filtre[['MCTS_Score_Moyen', 'Solveur_Score']].values
                
            fig.add_trace(go.Scatter(
                x=df_filtre.index,
                y=df_filtre['Opti_Moyen_%'],
                mode='markers',
                name='MCTS (% de l\'optimum ± Variance)',
                marker=dict(color='blue', size=8),
                customdata=custom_data,
                hovertemplate=hover_text,
                error_y=dict(
                    type='data',
                    symmetric=False,
                    array=df_filtre['Opti_Max_%'] - df_filtre['Opti_Moyen_%'], # Haut
                    arrayminus=df_filtre['Opti_Moyen_%'] - df_filtre['Opti_Min_%'], # Bas
                    color='rgba(0, 0, 255, 0.4)',
                    thickness=2,
                    width=4
                )
            ))
            
            titre = f"Optimalité et Variance MCTS — Agents: {ag} | Objets: {ob} | Ratio: {rat} | Graine: {sd}"
            
            # Définir l'échelle Y pour voir correctement les données (de 0 à un peu plus de 100)
            y_min = max(0, df_filtre['Opti_Min_%'].min() - 5)
            y_max = max(105, df_filtre['Opti_Max_%'].max() + 5)
            
            fig.update_layout(
                title=titre,
                xaxis_title="Instances testées",
                yaxis_title="Pourcentage du score optimal (%)",
                template="plotly_white",
                hovermode="closest",
                yaxis=dict(range=[y_min, y_max]),
                height=500,
                margin=dict(l=40, r=40, t=60, b=40)
            )
            
            fig.show()

    dropdown_agents.observe(update_plot, names='value')
    dropdown_objets.observe(update_plot, names='value')
    dropdown_seed.observe(update_plot, names='value')
    slider_ratio.observe(update_plot, names='value')
    
    display(ui, out)
    update_plot()

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import plotly.express as px
import pandas as pd

def plot_boxplot_interactive(df):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    # Normalisation : on calcule l'optimalité en %
    df_safe = df[df['Solveur_Score'] > 0].copy()
    df_safe['Opti_Moyen_%'] = (df_safe['MCTS_Score_Moyen'] / df_safe['Solveur_Score']) * 100
    df_safe['Opti_Moyen_%'] = df_safe['Opti_Moyen_%'].clip(upper=100) # Sécurité

    # 1. Widgets
    dropdown_y = widgets.Dropdown(
        options=[('Optimalité du MCTS (%)', 'Opti'), ('Temps d\'exécution (µs)', 'Temps')],
        value='Temps', # Je l'ai mis sur Temps par défaut pour vérifier que ton bug est résolu !
        description='📈 Analyser:',
        layout={'width': 'max-content'}
    )
    
    axe_x_options = [('Nombre d\'Agents', 'Agents'), ('Nombre d\'Objets', 'Objets')]
    if 'Ratio' in df_safe.columns:
        axe_x_options.append(('Ratio Random', 'Ratio'))
        
    dropdown_x = widgets.Dropdown(
        options=axe_x_options,
        value='Agents',
        description='📊 Grouper par:',
        layout={'width': 'max-content'}
    )

    ui = widgets.HBox([dropdown_y, dropdown_x])
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            col_y = dropdown_y.value
            col_x = dropdown_x.value
            
            df_plot = df_safe.copy()
            
            # --- CORRECTION 1 : Forcer les catégories ---
            # On ajoute le nom du filtre (ex: "3 Agents", "0.5 Ratio") pour forcer Plotly à séparer les boîtes
            suffixe = dropdown_x.label.split(" ")[-1]
            df_plot['X_Labels'] = df_plot[col_x].astype(str) + " " + suffixe
            
            # Tri logique des catégories 
            sorted_unique_vals = sorted(df_safe[col_x].unique())
            sorted_labels = [str(val) + " " + suffixe for val in sorted_unique_vals]
            df_plot['X_Labels'] = pd.Categorical(df_plot['X_Labels'], categories=sorted_labels, ordered=True)

            if col_y == 'Opti':
                # --- GRAPHIQUE 1 : Boîtes à moustaches de l'Optimalité ---
                fig = px.box(
                    df_plot,
                    x='X_Labels',
                    y="Opti_Moyen_%",
                    color='X_Labels',
                    title=f"Distribution de l'Optimalité du MCTS selon le {dropdown_x.label}",
                    labels={'X_Labels': dropdown_x.label, "Opti_Moyen_%": "Optimalité (%)"},
                    points="all", 
                    hover_data=["Solveur_Score", "MCTS_Score_Moyen"] + (["Seed"] if "Seed" in df_plot.columns else [])
                )
                
                fig.add_hline(y=100, line_dash="dash", line_color="red", annotation_text="Optimum (100%)")
                y_min = max(0, df_plot['Opti_Moyen_%'].min() - 5)
                fig.update_layout(yaxis=dict(range=[y_min, 105]), showlegend=False)
                
            else: 
                # --- GRAPHIQUE 2 : Comparaison Solveur vs MCTS sur le Temps ---
                colonnes_a_garder = ['X_Labels'] + (['Seed'] if 'Seed' in df_plot.columns else [])
                df_melt = df_plot.melt(
                    id_vars=colonnes_a_garder,
                    value_vars=['Solveur_Temps_us', 'MCTS_Temps_Moyen_us'],
                    var_name='Algorithme',
                    value_name='Temps_us'
                )
                
                df_melt['Algorithme'] = df_melt['Algorithme'].replace({
                    'Solveur_Temps_us': 'Solveur Exact',
                    'MCTS_Temps_Moyen_us': 'MCTS (Moyen)'
                })
                
                # --- CORRECTION 2 : Éviter le crash de l'échelle logarithmique ---
                # Si le temps est de 0, on le met à 1 µs artificiellement. 
                df_melt['Temps_us'] = df_melt['Temps_us'].clip(lower=1)
                
                fig = px.box(
                    df_melt,
                    x='X_Labels',
                    y="Temps_us",
                    color="Algorithme",
                    title=f"Comparaison des Temps d'exécution selon le {dropdown_x.label}",
                    labels={'X_Labels': dropdown_x.label, "Temps_us": "Temps d'exécution (µs)"},
                    points="outliers", 
                    color_discrete_sequence=["red", "blue"],
                    hover_data=["Seed"] if "Seed" in df_melt.columns else []
                )
                
                fig.update_yaxes(type="log") 
                fig.update_layout(boxmode="group") 

            # Mise en page finale et forçage de l'axe X en catégories
            fig.update_layout(template="plotly_white", margin=dict(t=60, b=40, l=40, r=40))
            fig.update_xaxes(type='category') # Interdit formellement à Plotly de faire une courbe continue
            fig.show()

    # 2. Écouteurs d'événements
    dropdown_y.observe(update_plot, names='value')
    dropdown_x.observe(update_plot, names='value')
    
    # 3. Affichage
    display(ui, out)
    update_plot()

In [ ]:
def plot_nuage_points_metriques_interactive(df):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    # 1. Préparation et Normalisation des données
    df_plot = df[df['Solveur_Score'] > 0].copy()
    df_plot['Optimalite_MCTS_%'] = (df_plot['MCTS_Score_Moyen'] / df_plot['Solveur_Score']) * 100
    df_plot['Optimalite_MCTS_%'] = df_plot['Optimalite_MCTS_%'].clip(upper=100)

    # 2. Création des Profils Combinés (Signatures)
    # On considère qu'une métrique est "validée" si le MCTS la trouve dans au moins 50% de ses essais (>= 0.5)
    def get_signature(row):
        sigs = []
        if row.get('MCTS_ParetoOptimal', 0) >= 0.5: 
            sigs.append('PO')
            
        # Logique d'inclusion : EF > EFX > EF1
        if row.get('MCTS_EF', 0) >= 0.5: 
            sigs.append('EF')
        elif row.get('MCTS_EFX', 0) >= 0.5: 
            sigs.append('EFX')
        elif row.get('MCTS_EF1', 0) >= 0.5: 
            sigs.append('EF1')
            
        if row.get('MCTS_Prop', 0) >= 0.5: 
            sigs.append('Prop')
            
        return " + ".join(sigs) if sigs else "Aucune"

    df_plot['Profil_Equite'] = df_plot.apply(get_signature, axis=1)

    # 3. Listes des options pour les widgets
    agents_options = ['Tous'] + sorted(df_plot['Agents'].unique().tolist())
    objets_options = ['Tous'] + sorted(df_plot['Objets'].unique().tolist())
    ratio_options = ['Tous'] + sorted(df_plot['Ratio'].unique().tolist()) if 'Ratio' in df_plot.columns else ["Non défini"]
    
    axes_options = [
        ('Optimalité MCTS (%)', 'Optimalite_MCTS_%'),
        ('Score Optimal (Solveur)', 'Solveur_Score'),
        ('Score Moyen MCTS', 'MCTS_Score_Moyen'),
        ("Temps d'exécution MCTS (µs)", 'MCTS_Temps_Moyen_us'),
        ("Ratio Random", 'Ratio')
    ]
    
    metriques_dispos = [m.replace('MCTS_', '') for m in df_plot.columns if m.startswith('MCTS_') and m.replace('MCTS_', '') in ["ParetoOptimal", "Prop", "EF", "EFX", "EF1"]]
    couleur_options = [('💡 Combinaison globale (Profil)', 'Profil_Equite')] + [(f"Métrique seule : {m}", f"MCTS_{m}") for m in metriques_dispos]

    # 4. Création des widgets
    # --- Ligne 1 : Filtres ---
    dd_agents = widgets.Dropdown(options=agents_options, value='Tous', description='👥 Agents:', layout={'width': 'max-content'})
    dd_objets = widgets.Dropdown(options=objets_options, value='Tous', description='📦 Objets:', layout={'width': 'max-content'})
    slider_ratio = widgets.SelectionSlider(options=ratio_options, value='Tous', description='🎲 Ratio:', continuous_update=False, layout={'width': '350px'})
    
    # --- Ligne 2 : Configuration du Graphique ---
    dd_x = widgets.Dropdown(options=axes_options, value='Solveur_Score', description='➡️ Axe X:', layout={'width': 'max-content'})
    dd_y = widgets.Dropdown(options=axes_options, value='Optimalite_MCTS_%', description='⬆️ Axe Y:', layout={'width': 'max-content'})
    dd_couleur = widgets.Dropdown(options=couleur_options, value='Profil_Equite', description='🎨 Couleur:', layout={'width': 'max-content'})

    ui_filtres = widgets.HBox([dd_agents, dd_objets, slider_ratio])
    ui_axes = widgets.HBox([dd_x, dd_y, dd_couleur])
    ui = widgets.VBox([ui_filtres, ui_axes])
    
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            
            # Filtrage des données
            df_filtre = df_plot.copy()
            if dd_agents.value != 'Tous': df_filtre = df_filtre[df_filtre['Agents'] == dd_agents.value]
            if dd_objets.value != 'Tous': df_filtre = df_filtre[df_filtre['Objets'] == dd_objets.value]
            if slider_ratio.value != 'Tous' and slider_ratio.value != "Non défini": df_filtre = df_filtre[df_filtre['Ratio'] == slider_ratio.value]

            if df_filtre.empty:
                print("⚠️ Aucune donnée pour cette sélection.")
                return

            col_x = dd_x.value
            col_y = dd_y.value
            col_c = dd_couleur.value

            # Configuration des infobulles (Hover Data)
            hover_dict = {
                col_x: ':.2f', col_y: ':.2f',
                "Agents": True, "Objets": True, "Ratio": True,
                "Profil_Equite": True
            }
            # On ajoute le détail des pourcentages des métriques au survol
            for m in metriques_dispos:
                hover_dict[f"MCTS_{m}"] = ':.2f'

            # --- DESSIN DU GRAPHIQUE ---
            if col_c == 'Profil_Equite':
                # Mode Combinaison (Couleurs discrètes)
                fig = px.scatter(
                    df_filtre, x=col_x, y=col_y, color=col_c,
                    title="Analyse Multidimensionnelle des Allocations MCTS",
                    labels={col_x: dd_x.label, col_y: dd_y.label, col_c: "Profil d'Équité atteint"},
                    hover_data=hover_dict,
                    opacity=0.8,
                    color_discrete_sequence=px.colors.qualitative.Prism # Palette bien distincte
                )
            else:
                # Mode Métrique Unique (Dégradé de couleur Rouge -> Vert)
                fig = px.scatter(
                    df_filtre, x=col_x, y=col_y, color=col_c,
                    title=f"Réussite de la métrique {col_c.replace('MCTS_', '')} par le MCTS",
                    labels={col_x: dd_x.label, col_y: dd_y.label, col_c: "Taux de réussite (0 à 1)"},
                    hover_data=hover_dict,
                    color_continuous_scale="RdYlGn", # Rouge (0) vers Jaune puis Vert (1)
                    range_color=[0, 1], # On force l'échelle de 0% à 100%
                    opacity=0.8
                )

            # Esthétique générale
            fig.update_traces(marker=dict(size=9, line=dict(width=1, color='DarkSlateGrey')))
            
            # Si on affiche l'optimalité en Y, on met la ligne de référence à 100%
            if col_y == 'Optimalite_MCTS_%':
                fig.add_hline(y=100, line_dash="dash", line_color="red", annotation_text="Optimum (100%)")
                fig.update_layout(yaxis=dict(range=[max(0, df_filtre[col_y].min() - 5), 105]))

            # Si on affiche le temps en Y ou X, on passe en log
            if 'Temps' in col_y: fig.update_yaxes(type="log")
            if 'Temps' in col_x: fig.update_xaxes(type="log")

            fig.update_layout(template="plotly_white", margin=dict(t=60, b=40, l=40, r=40), height=600)
            fig.show()

    # 5. Liaisons
    dd_agents.observe(update_plot, names='value')
    dd_objets.observe(update_plot, names='value')
    slider_ratio.observe(update_plot, names='value')
    dd_x.observe(update_plot, names='value')
    dd_y.observe(update_plot, names='value')
    dd_couleur.observe(update_plot, names='value')

    display(ui, out)
    update_plot()

In [ ]:
def plot_nuage_points_3d_interactive(df):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    # 1. Préparation et Normalisation des données
    df_plot = df[df['Solveur_Score'] > 0].copy()
    df_plot['Optimalite_MCTS_%'] = (df_plot['MCTS_Score_Moyen'] / df_plot['Solveur_Score']) * 100
    df_plot['Optimalite_MCTS_%'] = df_plot['Optimalite_MCTS_%'].clip(upper=100)

    # 2. Création des Profils Combinés (Signatures)
    def get_signature(row):
        sigs = []
        if row.get('MCTS_ParetoOptimal', 0) >= 0.5: 
            sigs.append('PO')
            
        if row.get('MCTS_EF', 0) >= 0.5: 
            sigs.append('EF')
        elif row.get('MCTS_EFX', 0) >= 0.5: 
            sigs.append('EFX')
        elif row.get('MCTS_EF1', 0) >= 0.5: 
            sigs.append('EF1')
            
        if row.get('MCTS_Prop', 0) >= 0.5: 
            sigs.append('Prop')
            
        return " + ".join(sigs) if sigs else "Aucune"

    df_plot['Profil_Equite'] = df_plot.apply(get_signature, axis=1)

    # 3. Listes des options pour les widgets
    agents_options = ['Tous'] + sorted(df_plot['Agents'].unique().tolist())
    objets_options = ['Tous'] + sorted(df_plot['Objets'].unique().tolist())
    
    axes_options = [
        ('Optimalité MCTS (%)', 'Optimalite_MCTS_%'),
        ('Score Optimal (Solveur)', 'Solveur_Score'),
        ('Score Moyen MCTS', 'MCTS_Score_Moyen'),
        ("Temps d'exécution MCTS (µs)", 'MCTS_Temps_Moyen_us'),
        ("Ratio Random", 'Ratio')
    ]
    
    metriques_dispos = [m.replace('MCTS_', '') for m in df_plot.columns if m.startswith('MCTS_') and m.replace('MCTS_', '') in ["ParetoOptimal", "Prop", "EF", "EFX", "EF1"]]
    couleur_options = [('💡 Combinaison globale (Profil)', 'Profil_Equite')] + [(f"Métrique seule : {m}", f"MCTS_{m}") for m in metriques_dispos]

    # 4. Création des widgets
    # --- Ligne 1 : Filtres ---
    dd_agents = widgets.Dropdown(options=agents_options, value='Tous', description='👥 Agents:', layout={'width': 'max-content'})
    dd_objets = widgets.Dropdown(options=objets_options, value='Tous', description='📦 Objets:', layout={'width': 'max-content'})
    
    # --- Ligne 2 : Configuration du Graphique 3D ---
    dd_x = widgets.Dropdown(options=axes_options, value='Solveur_Score', description='➡️ Axe X:', layout={'width': 'max-content'})
    dd_y = widgets.Dropdown(options=axes_options, value='Optimalite_MCTS_%', description='⬆️ Axe Y:', layout={'width': 'max-content'})
    dd_z = widgets.Dropdown(options=axes_options, value='Ratio', description='↗️ Axe Z:', layout={'width': 'max-content'})
    
    # --- Ligne 3 : Couleur ---
    dd_couleur = widgets.Dropdown(options=couleur_options, value='Profil_Equite', description='🎨 Couleur:', layout={'width': 'max-content'})

    ui_filtres = widgets.HBox([dd_agents, dd_objets])
    ui_axes = widgets.HBox([dd_x, dd_y, dd_z])
    ui_couleur = widgets.HBox([dd_couleur])
    ui = widgets.VBox([ui_filtres, ui_axes, ui_couleur])
    
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            
            # Filtrage des données (Uniquement Agents et Objets car le Ratio est sur Z)
            df_filtre = df_plot.copy()
            if dd_agents.value != 'Tous': df_filtre = df_filtre[df_filtre['Agents'] == dd_agents.value]
            if dd_objets.value != 'Tous': df_filtre = df_filtre[df_filtre['Objets'] == dd_objets.value]

            if df_filtre.empty:
                print("⚠️ Aucune donnée pour cette sélection.")
                return

            col_x = dd_x.value
            col_y = dd_y.value
            col_z = dd_z.value
            col_c = dd_couleur.value

            # Configuration des infobulles (Hover Data)
            hover_dict = {
                col_x: ':.2f', col_y: ':.2f', col_z: ':.2f',
                "Agents": True, "Objets": True, "Ratio": True,
                "Profil_Equite": True
            }
            for m in metriques_dispos:
                hover_dict[f"MCTS_{m}"] = ':.2f'

            # --- DESSIN DU GRAPHIQUE 3D ---
            if col_c == 'Profil_Equite':
                fig = px.scatter_3d(
                    df_filtre, x=col_x, y=col_y, z=col_z, color=col_c,
                    title="Analyse Multidimensionnelle 3D des Allocations MCTS",
                    labels={col_x: dd_x.label, col_y: dd_y.label, col_z: dd_z.label, col_c: "Profil d'Équité"},
                    hover_data=hover_dict,
                    opacity=0.8,
                    color_discrete_sequence=px.colors.qualitative.Prism 
                )
            else:
                fig = px.scatter_3d(
                    df_filtre, x=col_x, y=col_y, z=col_z, color=col_c,
                    title=f"Réussite de la métrique {col_c.replace('MCTS_', '')} par le MCTS (Vue 3D)",
                    labels={col_x: dd_x.label, col_y: dd_y.label, col_z: dd_z.label, col_c: "Taux de réussite (0 à 1)"},
                    hover_data=hover_dict,
                    color_continuous_scale="RdYlGn", 
                    range_color=[0, 1], 
                    opacity=0.8
                )

            # Esthétique générale 3D (Marqueurs plus petits pour mieux voir en profondeur)
            fig.update_traces(marker=dict(size=4, line=dict(width=0)))
            
            # Gestion dynamique des échelles logarithmiques pour les axes 3D
            if 'Temps' in col_x: fig.update_layout(scene_xaxis_type="log")
            if 'Temps' in col_y: fig.update_layout(scene_yaxis_type="log")
            if 'Temps' in col_z: fig.update_layout(scene_zaxis_type="log")

            fig.update_layout(
                template="plotly_white", 
                margin=dict(t=60, b=0, l=0, r=0), 
                height=750,
                scene=dict(
                    xaxis_title=dd_x.label,
                    yaxis_title=dd_y.label,
                    zaxis_title=dd_z.label
                )
            )
            fig.show()

    # 5. Liaisons
    dd_agents.observe(update_plot, names='value')
    dd_objets.observe(update_plot, names='value')
    dd_x.observe(update_plot, names='value')
    dd_y.observe(update_plot, names='value')
    dd_z.observe(update_plot, names='value')
    dd_couleur.observe(update_plot, names='value')

    display(ui, out)
    update_plot()

In [ ]:
# Exécution
nom_du_fichier_zip = "../" + "results/experiments_17-06-2026_10-02-59.zip"
df = parse_zip_experiments(nom_du_fichier_zip)

100%|██████████| 101102/101102 [01:49<00:00, 925.09it/s]


In [ ]:
#plot_comparisons(df)

In [ ]:
#plot_impact_ratio(df)

In [ ]:
#plot_ratio_impact_on_score(df)

In [ ]:
#plot_erreur_normalisee(df)

In [ ]:

""" 
print("---")
plot_impact_ratio_interactive(df, show_all=False)
print("---")
plot_ratio_impact_on_score_interactive(df, show_all=False)
print("---")
plot_erreur_normalisee_interactive(df, show_all=False)
print("---")
plot_metrics_comparison_interactive(df) 
print("---")
widget_variance_mcts_normalise(df)
 print("---") 
plot_boxplot_interactive(df) 
print("---") 
plot_nuage_points_metriques_interactive(df)
plot_nuage_points_3d_interactive(df)
"""
plot_comparisons_interactive(df)

Output()